# Filling a Form: The Agent Acts [Agent Patterns - Module 10]

> **MLCourse - Agentic AI - Agent Patterns**

Reading is safe. **Acting** is where browser agents become both useful and
frightening: the same mechanism that files a support ticket can also cancel
a subscription, send an email, or place an order.

This notebook builds the smallest honest version of an acting agent, against
a local form, and shows the two design decisions that keep it safe:

- the agent picks **values**, never **selectors**;
- every action goes through a **whitelist** of allowed operations.

### What you will learn

1. Playwright's action API: `fill`, `select_option`, `check`, `click`.
2. Giving the model a *field spec* and asking only for values.
3. An allow-listed action executor - the key safety pattern.
4. Reading the result back to confirm the action actually happened.

### Key takeaways

- Separate *decide* (model) from *execute* (your code).
- Never build a selector out of model output.
- Verify the outcome; a click that silently did nothing is the norm.

### Setup: imports, environment, track discovery


In [ ]:
import os
import re
import sys
import json
import time
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

HERE = Path.cwd().resolve()                  # the module folder
FIXTURES = HERE / "fixtures"
FIXTURES.mkdir(exist_ok=True)

print(f"Track root : {TRACK}")
print(f"Module dir : {HERE}")
print(f"Model      : {MODEL} (via Groq)")


### Running Playwright inside Jupyter (Windows)


In [ ]:
# Two environment quirks, handled once here and reused in every notebook:
#
# 1. Jupyter's kernel already runs an asyncio event loop in the main thread.
#    Playwright's SYNC api refuses to start inside a running loop, so we run
#    every browser job in a short-lived worker thread.
# 2. ipykernel installs the Selector event-loop policy on Windows, and that
#    policy cannot spawn subprocesses - which is exactly what launching
#    Chromium needs. We restore the Proactor policy so new loops can.
#
# In a plain .py script neither applies: `with sync_playwright() as p:` just works.

import asyncio
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

def browse(url, job, headless=True):
    """Open `url` in Chromium, hand the Page to job(page), return its result."""
    def _run():
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=headless)
            page = browser.new_page()
            page.goto(url)
            try:
                return job(page)
            finally:
                browser.close()          # never leak a Chromium process
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(_run).result()

print("browse() ready - policy:", type(asyncio.get_event_loop_policy()).__name__)


### One small Groq client, with 429 backoff


In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def ask(prompt, system="You are a precise assistant.", max_tokens=600, temperature=0.0):
    """One chat completion, with exponential backoff for the free tier."""
    for attempt in range(5):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            wait = 2 ** attempt + random.random()
            print(f"  retry {attempt+1} in {wait:.1f}s ({type(e).__name__})")
            time.sleep(wait)
    raise RuntimeError("Groq call failed after 5 attempts")

print("Groq helper ready.")


### Write the local form fixture


In [ ]:
FORM = FIXTURES / "contact.html"
FORM.write_text("""<!doctype html>
<html><head><meta charset="utf-8"><title>Northwind Parts - Contact</title></head>
<body>
  <h1>Contact us</h1>
  <p>Tell us what you need and we will reply within one working day.</p>

  <form id="contact-form">
    <label for="name">Full name</label>
    <input id="name" name="name" type="text">

    <label for="email">Email</label>
    <input id="email" name="email" type="email">

    <label for="topic">Topic</label>
    <select id="topic" name="topic">
      <option value="">-- choose --</option>
      <option value="order">Order status</option>
      <option value="stock">Stock enquiry</option>
      <option value="returns">Returns</option>
      <option value="other">Something else</option>
    </select>

    <label for="message">Message</label>
    <textarea id="message" name="message" rows="4"></textarea>

    <label><input id="urgent" name="urgent" type="checkbox"> Mark as urgent</label>

    <button id="submit" type="button">Send</button>
  </form>

  <div id="result"></div>

  <script>
    document.getElementById('submit').addEventListener('click', function () {
      var f = document.getElementById('contact-form');
      var missing = [];
      ['name', 'email', 'topic', 'message'].forEach(function (k) {
        if (!f[k].value) missing.push(k);
      });
      var out = document.getElementById('result');
      if (missing.length) {
        out.textContent = 'ERROR: missing fields: ' + missing.join(', ');
        return;
      }
      out.textContent = 'TICKET-4471 created for ' + f.name.value +
        ' (' + f.topic.value + (f.urgent.checked ? ', urgent' : '') + ')';
    });
  </script>
</body></html>
""", encoding="utf-8")

print("wrote", FORM)


### 1. The action API

Four calls cover most forms:

```python
page.fill("#name", "Priya Raman")        # text input / textarea
page.select_option("#topic", "stock")    # <select>, by VALUE not label
page.check("#urgent")                    # checkbox / radio
page.click("#submit")                    # buttons and links
```

Each auto-waits for the element to exist and be actionable, so you do not
write sleeps. `select_option` takes the option's `value` attribute by
default - a classic first-time bug is passing the visible label.

Let us do it manually once, so the agent version has nothing mysterious in it.

### Fill the form by hand


In [ ]:
def fill_by_hand(page):
    page.fill("#name", "Priya Raman")
    page.fill("#email", "priya@example.com")
    page.select_option("#topic", "stock")
    page.fill("#message", "Do you expect NW-1002 back in stock this week?")
    page.check("#urgent")
    page.click("#submit")
    return page.inner_text("#result")

print("page said:", browse(FORM.as_uri(), fill_by_hand))


### What failure looks like


In [ ]:
# Leave required fields blank. Note that the CLICK still succeeds - the page
# just refuses. This is why you must read the result back.

def incomplete(page):
    page.fill("#name", "Priya Raman")
    page.click("#submit")          # no exception raised!
    return page.inner_text("#result")

print("page said:", browse(FORM.as_uri(), incomplete))
print()
print("Playwright reported success. The page reported failure.")
print("An agent that only checks for exceptions would think this worked.")


### 2. The model decides values, not selectors

Here is the design decision that matters most.

**Bad:** ask the model for a list of `(selector, action, value)` triples and
execute them. You have now given an LLM arbitrary control of a browser, and
the page's own text can influence what it emits (notebook 04).

**Good:** you enumerate the fields yourself - you wrote the automation, you
know the form - and ask the model only for the *values*. The model's output
space is data, not code.

### Describe the form to the model


In [ ]:
# This spec is written by YOU from the page, not generated by the model.

FIELD_SPEC = [
    {"key": "name",    "kind": "text",     "desc": "customer full name"},
    {"key": "email",   "kind": "text",     "desc": "customer email address"},
    {"key": "topic",   "kind": "select",   "desc": "one of: order, stock, returns, other"},
    {"key": "message", "kind": "text",     "desc": "the enquiry, max 2 sentences"},
    {"key": "urgent",  "kind": "checkbox", "desc": "true only if the customer is blocked"},
]

# The selector for each key also lives in YOUR code. The model never sees it.
SELECTORS = {
    "name": "#name", "email": "#email", "topic": "#topic",
    "message": "#message", "urgent": "#urgent",
}

for f in FIELD_SPEC:
    print(f"  {f['key']:9s} {f['kind']:9s} {f['desc']}")


### Ask for values only


In [ ]:
REQUEST = ("Priya Raman (priya@example.com) needs to know when the copper tee "
           "NW-1002 comes back in stock. Her production line is stopped until "
           "it arrives.")

VALUES_PROMPT = """You are filling a web contact form on behalf of a customer.

FIELDS (fill every one):
{spec}

CUSTOMER REQUEST:
{request}

Return ONLY a JSON object mapping each field key to its value.
Use true/false for checkbox fields. For "topic" use exactly one of the
allowed values. No prose, no markdown fence."""

raw = ask(VALUES_PROMPT.format(
    spec="\n".join(f"- {f['key']} ({f['kind']}): {f['desc']}" for f in FIELD_SPEC),
    request=REQUEST,
), system="You output JSON and nothing else.")

print(raw)


### Validate the values BEFORE touching the browser


In [ ]:
def parse_json(text):
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    if fence:
        text = fence.group(1).strip()
    s, e = text.find("{"), text.rfind("}")
    return json.loads(text[s:e + 1])

ALLOWED_TOPICS = {"order", "stock", "returns", "other"}

values = parse_json(raw)

# 1. every declared field present, and nothing else
keys = {f["key"] for f in FIELD_SPEC}
assert set(values) == keys, f"key mismatch: got {set(values)}, want {keys}"
# 2. constrained field inside its domain
assert values["topic"] in ALLOWED_TOPICS, f"bad topic: {values['topic']}"
# 3. types match the declared kind
assert isinstance(values["urgent"], bool), "urgent must be a boolean"

print("validated values:")
for k, v in values.items():
    print(f"  {k:9s} {v!r}")


Three assertions, three real failure modes: hallucinated extra fields, an
out-of-domain enum value, a string `"true"` where a boolean was required.
Each would have produced a confusing Playwright error deep in the run. Catch
them at the boundary, where the message is clear.

### 3. The allow-listed executor

The executor is the security boundary. It accepts a **key** from a fixed set
and a **value**, looks the selector up in your own table, and dispatches to
one of three hard-coded operations. There is no path from model output to an
arbitrary selector or an arbitrary Playwright method.

### The executor


In [ ]:
KINDS = {f["key"]: f["kind"] for f in FIELD_SPEC}

def apply_value(page, key, value):
    """Apply ONE validated value. Whitelisted keys and operations only."""
    if key not in SELECTORS:
        raise ValueError(f"refusing unknown field: {key!r}")
    sel, kind = SELECTORS[key], KINDS[key]

    if kind == "text":
        page.fill(sel, str(value))
    elif kind == "select":
        page.select_option(sel, str(value))
    elif kind == "checkbox":
        page.check(sel) if value else page.uncheck(sel)
    else:
        raise ValueError(f"unsupported kind: {kind!r}")
    return f"{kind}({key}) <- {value!r}"

# The executor refuses anything not on the list, whatever the model says.
try:
    apply_value(None, "admin_override", True)
except ValueError as e:
    print("blocked:", e)


### Run the agent end to end


In [ ]:
def agent_run(page):
    log = []
    for key in [f["key"] for f in FIELD_SPEC]:       # fixed order, our order
        log.append(apply_value(page, key, values[key]))

    page.click("#submit")                            # the one hard-coded action
    outcome = page.inner_text("#result")

    # Read the fields back: did the page actually take what we set?
    readback = {
        "name": page.input_value("#name"),
        "topic": page.input_value("#topic"),
        "urgent": page.is_checked("#urgent"),
    }
    return log, outcome, readback

log, outcome, readback = browse(FORM.as_uri(), agent_run)

print("actions:")
for line in log:
    print("  ", line)
print("\nresult  :", outcome)
print("readback:", readback)
print()
success = outcome.startswith("TICKET-")
print("SUCCESS" if success else "FAILED")
assert success, f"the form did not accept the submission: {outcome}"


### Verify, do not assume

Three checks in that cell, and every one earns its place:

- the **result text** - the page's own verdict,
- the **readback** - what the DOM actually holds, which catches a `fill` that
  targeted the wrong element or was overwritten by page JavaScript,
- an **assert** - so a failed submission stops the pipeline instead of
  flowing on as a silent no-op.

### Pitfalls recap

- **Letting the model emit selectors.** It is code injection wearing a
  friendly hat. Enumerate the fields yourself.
- **Passing option labels to `select_option`.** It wants the `value`.
- **Assuming click == success.** Read the outcome back, always.
- **Acting on a page you have not read.** Which is exactly notebook 04.
- **No dry-run mode.** For anything destructive, log the planned actions and
  require confirmation before executing. See module 14 of this track.

### Next

Notebook 04: the page text is written by an attacker. Everything above still
runs - and that is the problem.